# Занятие 16. Практика: разброс, аномалии и проверка гипотез

> **Цель занятия:** провести настоящее расследование по данным. Заказчик задаёт вопрос,
> а мы ищем ответ: чистим аномалии, смотрим на распределения через **boxplot** и **violin**,
> проверяем догадки **t-тестом**, **критерием Манна–Уитни** и **χ²**.

**Что делаем сегодня:**
1. Разбираемся, что за вопрос у заказчика и в какой метрике лежат его деньги;
2. Ищем аномалии в данных — правилом **IQR** и по **Z-score** — и выясняем их **природу**;
3. Смотрим на распределения по группам и находим подозреваемого;
4. Проверяем догадки **тестами** и даём заказчику ответ, подкреплённый p-value.

Практика — **10 заданий, 30 баллов**. Пишите код в ячейках после `# Ваш код`, а там, где
просят, — короткий текст после `**Вывод:**`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
sns.set_theme(style='whitegrid')

---
## Вопрос заказчика

К вам пришёл руководитель сервиса доставки еды. Он говорит так:

> «Заказов у нас много, выручка растёт. Но **клиенты не возвращаются**: половина делает
> один заказ и пропадает. Каждого следующего клиента мы покупаем рекламой заново, и это
> съедает всю прибыль. Мы уже потратились на **промокоды** — не помогло.
>
> **Что мешает нам зарабатывать?**»

Заказчик не говорит, где искать. Он даёт данные и метрику, в которой лежат его деньги:
**`repeat_order`** — вернулся ли клиент за вторым заказом. Наша задача — найти, **что** сбивает
эту метрику, и доказать это числами, а не ощущениями.

### Данные

Таблица `data/orders.csv` — **2000 заказов** за месяц.

| Столбец | Что означает |
|---|---|
| `order_id` | номер заказа |
| `zone` | район города: `Центр`, `Северный`, `Приморский`, `Заречный` |
| `courier_type` | кто вёз: `пеший`, `вело`, `авто` |
| `promo` | был ли промокод: `да` / `нет` |
| `delivery_time` | **время доставки в минутах** |
| `order_sum` | сумма заказа, рублей |
| `rating` | оценка клиента, 1–5 |
| `repeat_order` | **вернулся ли клиент**: `1` — сделал повторный заказ, `0` — нет |

Главная рабочая гипотеза очевидна: **людей бесит долгое ожидание**. Но прежде чем её проверять,
данные придётся привести в порядок — иначе тесты соврут.

---
## Что понадобится из теории

Коротко напомним инструменты занятия 15 — все задания решаются ими.

**Поиск выбросов.** Правило **1.5 · IQR** опирается на квартили и устойчиво к самим выбросам:

```python
Q1 = df['x'].quantile(0.25)
Q3 = df['x'].quantile(0.75)
IQR = Q3 - Q1
low, high = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
```

Правило **Z-score** смотрит, на сколько стандартных отклонений значение отстоит от среднего
(порог `|z| > 3`), но само страдает от выбросов — они раздувают `std`:

```python
z = (df['x'] - df['x'].mean()) / df['x'].std()
outliers = df[z.abs() > 3]
```

**Графики.** `sns.boxplot` показывает медиану, квартили, усы и точки-выбросы;
`sns.violinplot` добавляет к этому **форму** распределения и выдаёт несколько пиков.

**Тесты.** Выбор зависит от того, что сравниваем:

| Что сравниваем | Тест | Вызов |
|---|---|---|
| средние двух групп, данные ~нормальны | t-тест Стьюдента | `stats.ttest_ind(a, b)` |
| центры двух групп, есть скос и выбросы | Манна–Уитни | `stats.mannwhitneyu(a, b)` |
| доли (конверсия, повторные заказы) | χ² | `stats.chi2_contingency(table)` |

Решение всегда одно: **p-value < 0.05** — отвергаем H₀ (различие значимо);
**p-value ≥ 0.05** — улик недостаточно.

---
## Практика


### <font color='DarkOrange'>Задание 1 [2 балла]</font>

Загрузите таблицу `data/orders.csv` (разделитель `;`), выведите её размер и сводку
`.describe()` по столбцу `delivery_time`.

Посмотрите на `min`, `max`, `mean` и медиану (`50%`). Что в этих числах уже настораживает?

In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*

### <font color='DarkOrange'>Задание 2 [3 балла]</font>

Найдите выбросы в `delivery_time` по правилу **1.5 · IQR**: посчитайте `Q1`, `Q3`, `IQR`,
нижнюю и верхнюю границы нормы и выведите, **сколько** заказов вышло за эти границы.

In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 3 [3 балла]</font>

Найдите выбросы по **Z-score** (порог `|z| > 3`). Выведите их количество и **минимальное**
время доставки среди них. Сравните результат с заданием 2.

Почему правила разошлись так сильно? Подсказка: посмотрите на `std` и медиану.

In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*

### <font color='DarkOrange'>Задание 4 [3 балла]</font>

Разберитесь в **природе** аномалий — удалять вслепую нельзя. Отберите заказы с
`delivery_time < 5` и заказы с `delivery_time > 180` и посмотрите, **какие курьеры**
их выполняли (`value_counts()` по `courier_type` среди медленных).

Кто виноват в «зависших» заказах?

In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*

### <font color='DarkOrange'>Задание 5 [2 балла]</font>

Теперь примените то же правило **1.5 · IQR** к столбцу `order_sum`. Выведите верхнюю границу,
число заказов за ней и **сколько** из них дороже 8000 рублей (это корпоративные заказы на офис).

Надо ли удалять эти аномалии? Ответьте в выводе.

In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*

### <font color='DarkOrange'>Задание 6 [3 балла]</font>

Соберите чистую таблицу `clean`: оставьте только заказы с временем доставки **от 5 до 180**
минут включительно (`order_sum` не трогаем — там аномалий нет).

Выведите, сколько строк осталось, и сравните **среднее** время доставки до и после чистки.

In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*

### <font color='DarkOrange'>Задание 7 [3 балла]</font>

Найдите подозреваемого. Постройте по таблице `clean`:

- **boxplot** времени доставки по районам (`zone`);
- **violinplot** времени доставки по районам.

Какой район выбивается? Что violin показывает внутри него такого, чего не видно на boxplot?

In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*

### <font color='DarkOrange'>Задание 8 [4 балла]</font>

Проверим вторую догадку: **пешие курьеры медленнее велокурьеров**.

1. Сравните `t-тестом` время доставки пеших и вело **на исходной** таблице `orders`
   (со сбоями). Выведите средние и p-value.
2. Повторите **на чистой** таблице `clean`.
3. Запустите на **исходных** данных `критерий Манна–Уитни` и выведите медианы.

Сравните три результата. Почему t-тест на сырых данных дал такой ответ, а Манна–Уитни — нет?

In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*

### <font color='DarkOrange'>Задание 9 [4 балла]</font>

Пора ответить на главный вопрос: **бьёт ли долгая доставка по повторным заказам**.

На таблице `clean` создайте признак «долгая доставка» (`delivery_time > 45`), постройте
таблицу сопряжённости с `repeat_order` через `pd.crosstab`, посчитайте **долю повторных
заказов** в каждой группе и проверьте гипотезу **критерием χ²**.

Сформулируйте H₀ и вывод по p-value.

In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*

### <font color='DarkOrange'>Задание 10 [3 балла]</font>

Заказчик уверен, что дело в **промокодах** — на них уже потратились. Проверьте эту версию:
постройте на `clean` таблицу сопряжённости `promo` и `repeat_order`, посчитайте доли и
примените **χ²**.

В выводе дайте **финальный ответ заказчику**: что мешает зарабатывать и что делать.

In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*